# Lesson 4.1 – 4.3: Pehla ML model (T20 final score predictor)

**Sawal:** 10 overs ke baad ka score dekh ke, 20 overs ka **final score** predict karna.

Is notebook mein poora safar:

```
Dataset banana → Data samajhna → X / y → Train/test split
→ LinearRegression (sirf numbers) → Metrics → Residual analysis
→ OneHotEncoder (venue) → Behtar model → Weights padhna
```

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder

---
## Part 1: Practice dataset banana

12 rows ka data model ke liye bohat kam tha. Isliye 300 matches ka data **khud** banate hain, ek formula + thora random "shor" (noise) se.

**Faida:** hamein **asal formula pata hai** (answer key). Training ke baad check kar sakenge ke model ne woh formula **khud dhoonda** ya nahi. Real data mein ye kabhi pata nahi chalta.

**Asal formula (yaad rakhein):**
```
final_score ≈ 1.7 × runs_10  −  8 × wickets_10  +  30  +  venue bonus  +  noise
```
Venue bonus: Karachi **+5**, Lahore **0**, Rawalpindi **+15**, Multan **−8**

| Line | Matlab |
|---|---|
| `default_rng(42)` | random generator. 42 = seed, taake har dafa **same** data bane |
| `rng.choice(venues, n)` | 300 dafa randomly ek venue chuno |
| `rng.integers(55, 111, n)` | 10 overs ke runs: 55 se 110 |
| `rng.integers(0, 6, n)` | 10 overs ki wickets: 0 se 5 |
| `noise` | asli cricket ki be-tarteebi (±12 runs), jo **koi model predict nahi kar sakta** |

In [2]:
rng = np.random.default_rng(42)
n = 300

venues = ["Karachi", "Lahore", "Rawalpindi", "Multan"]
venue = rng.choice(venues, n)
runs_10 = rng.integers(55, 111, n)
wickets_10 = rng.integers(0, 6, n)

venue_bonus = {"Karachi": 5, "Lahore": 0, "Rawalpindi": 15, "Multan": -8}
bonus = np.array([venue_bonus[v] for v in venue])
noise = rng.normal(0, 12, n)

final_score = 1.7 * runs_10 + 30 - 8 * wickets_10 + bonus + noise

data = pd.DataFrame({
    "venue": venue,
    "runs_10": runs_10,
    "wickets_10": wickets_10,
    "final_score": final_score.round().astype(int),
})

data.to_csv("../data/t20_scores.csv", index=False)
data.head()

,venue,runs_10,wickets_10,final_score
0,Karachi,75,3,144
1,Multan,101,4,160
2,Rawalpindi,55,4,85
3,Lahore,105,0,220
4,Lahore,85,4,135


---
## Part 2: Data ko samajhna (har project ka pehla kaam)

In [3]:
df = pd.read_csv("../data/t20_scores.csv")
df.shape

(300, 4)

In [4]:
df.info()   # koi missing value? sab columns 300 non-null hone chahiye

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   venue        300 non-null    str  
 1   runs_10      300 non-null    int64
 2   wickets_10   300 non-null    int64
 3   final_score  300 non-null    int64
dtypes: int64(3), str(1)
memory usage: 9.5 KB


In [5]:
df.describe()   # min/max cricket ke hisaab se sense banate hain?

,runs_10,wickets_10,final_score
count,300.000000,300.000000,300.000000
mean,82.583333,2.573333,151.683333
std,17.162115,1.764938,35.695428
min,55.000000,0.000000,72.000000
25%,67.750000,1.000000,124.750000
50%,83.000000,3.000000,153.000000
75%,99.000000,4.000000,178.000000
max,110.000000,5.000000,238.000000


In [6]:
df["venue"].value_counts()   # har venue ke kitne matches?

venue
Lahore        87
Rawalpindi    74
Multan        71
Karachi       68
Name: count, dtype: int64

### Columns ka naqsha

| Column | Kya hai | Type | Role | Leakage? |
|---|---|---|---|---|
| `venue` | ground | categorical (nominal) | feature | ✅ match se pehle pata |
| `runs_10` | 10 overs ke runs | numerical | feature | ✅ 10 overs pe pata |
| `wickets_10` | 10 overs ki wickets | numerical | feature | ✅ 10 overs pe pata |
| `final_score` | 20 overs ka total | numerical | **target (y)** | – |

**Prediction kab ho rahi hai?** 10 overs ke baad. Teeno features us waqt maloom hain → **koi leakage nahi**.

---
## Part 3: X aur y

- **X** = features → **double brackets** → 2D table
- **y** = target → **single bracket** → 1D

In [7]:
X = df[["venue", "runs_10", "wickets_10"]]
y = df["final_score"]

print("X:", X.shape)   # (300, 3)
print("y:", y.shape)   # (300,)

X: (300, 3)
y: (300,)


---
## Part 4: Train/test split

**Exam ki misaal:** practice ke sawal (train) se seekho, final exam (test) mein **naye** sawal. Agar exam mein wahi sawal aayein to pata nahi chalega ke student ne **seekha** ya **ratta** lagaya.

- **Train (80%)**: model inhi se seekhta hai
- **Test (20%)**: model ne ye **kabhi nahi dekhe**, asli imtihan

| Parameter | Matlab |
|---|---|
| `test_size=0.2` | 20% test → 300 × 0.2 = **60** rows |
| `random_state=42` | shuffle har dafa **same**, taake results compare ho sakein |

⚠️ **Wapsi ka order fix hai:** `X_train, X_test, y_train, y_test` (pehle dono X, phir dono y).

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)   # (240, 3)
print("X_test: ", X_test.shape)    # (60, 3)
print("y_train:", y_train.shape)   # (240,)
print("y_test: ", y_test.shape)    # (60,)

X_train: (240, 3)
X_test:  (60, 3)
y_train: (240,)
y_test:  (60,)


### Alignment check
`X_train` aur `y_train` ke **index numbers** same, same order mein hone chahiye (shuffled, 0-1-2 nahi). Yahi saboot hai ke X aur y saath kate.

In [9]:
print(X_train.head())
print(y_train.head())

          venue  runs_10  wickets_10
232     Karachi       63           2
59   Rawalpindi       96           1
6       Karachi       71           1
185      Lahore       68           3
173  Rawalpindi       81           4
232    122
59     213
6      135
185    130
173    143
Name: final_score, dtype: int64


### Jaan boojh ke ghalti: order ulta likhna
Koi error nahi aata, lekin variables mein **ghalat cheezein** chali jati hain. Shapes dekh ke pakdein.

In [10]:
a, b, c, d = train_test_split(X, y, test_size=0.2, random_state=42)

# Agar koi likh de: X_train, y_train, X_test, y_test = ...
print("'X_train' asal mein:", a.shape, " → X_train ✅")
print("'y_train' asal mein:", b.shape, " → ye X_test hai ❌ (jawab ki jagah sawal)")
print("'X_test'  asal mein:", c.shape, "   → ye y_train hai ❌")
print("'y_test'  asal mein:", d.shape, "    → y_test ✅")

'X_train' asal mein: (240, 3)  → X_train ✅
'y_train' asal mein: (60, 3)  → ye X_test hai ❌ (jawab ki jagah sawal)
'X_test'  asal mein: (240,)    → ye y_train hai ❌
'y_test'  asal mein: (60,)     → y_test ✅


**Sabaq:** split ke baad **hamesha shapes print karein**.

---
## Part 5: Pehla model (sirf numeric features)

Abhi `venue` **chhor** rahe hain (text hai, Part 8 mein dekhenge kyun masla hai).

```
model = LinearRegression()     → khali model
model.fit(X, y)                → seekho (sab se achhe w aur b dhoondo)
model.predict(X_naya)          → ŷ = w₁x₁ + w₂x₂ + b
```
Lesson 1.3 mein aapne `w` aur `b` **haath se** badal ke MSE kam kiya tha. `fit()` yahi kaam **khud, sab se behtar** karta hai.

In [11]:
num_cols = ["runs_10", "wickets_10"]

model = LinearRegression()
model.fit(X_train[num_cols], y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](2,)","[ 1.65,-8.61]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](2,)","['runs_10','wickets_10']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,37.75
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,2
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(2)


### Model ne kya seekha?
- `coef_` = har feature ka weight (w), `num_cols` ke **same order** mein
- `intercept_` = b

In [12]:
print("Weights:  ", model.coef_.round(2))
print("Intercept:", round(model.intercept_, 2))

Weights:   [ 1.65 -8.61]
Intercept: 37.75


### Natija: model ne formula khud dhoond liya!

| | Asal formula | Model |
|---|---|---|
| runs_10 | **1.7** | **~1.65** |
| wickets_10 | **−8** | **~−8.6** |
| intercept | ~30 + venue ka average bonus (~3) ≈ **33** | **~37.8** |

Kisi ne model ko formula **nahi bataya**. Sirf 240 matches dekh ke nikal liya. **Training ka asal matlab yahi hai.**

Exact kyun nahi? **Noise** + sirf 240 examples + model `venue` nahi dekh pa raha (us ka asar intercept mein chhup gaya).

---
## Part 6: Test pe prediction aur metrics

In [13]:
y_pred = model.predict(X_test[num_cols])

comparison = pd.DataFrame({
    "actual": y_test,
    "predicted": y_pred.round(),
})
comparison.head(10)

,actual,predicted
203,176,165.0
266,100,114.0
152,149,173.0
9,96,95.0
233,140,153.0
226,172,178.0
196,110,95.0
109,102,111.0
5,97,114.0
175,166,156.0


| Metric | Cricket mein matlab |
|---|---|
| **MAE** | model **average kitne runs** se chookta hai |
| **RMSE** | MAE jaisa, lekin **bari ghaltiyon ko zyada saza**. Ye bhi runs mein |
| **R²** | final score ke utar charhao ka **kitna hissa** model samjha (1.0 = sab) |

**Train vs test R²:** qareeb → model ne **seekha** ✅ · train bohat zyada, test kam → **overfitting** (ratta)

In [14]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
r2_train = r2_score(y_train, model.predict(X_train[num_cols]))

print(f"MAE:      {mae:.2f}")
print(f"RMSE:     {rmse:.2f}")
print(f"Test R²:  {r2:.3f}")
print(f"Train R²: {r2_train:.3f}")

MAE:      13.00
RMSE:     15.50
Test R²:  0.777
Train R²: 0.838


### Natija
- MAE ~**13**: aam tor pe ±13 runs ki ghalti
- Train R² (~0.84) aur test R² (~0.78) **qareeb** → seekha hai, ratta nahi ✅

**R² 1.0 kyun nahi?** (1) **Noise** jo koi model nahi pakar sakta, aur (2) **venue gayab** hai. Pehle ka ilaaj nahi, **doosre ka hai** (Part 8).

---
## Part 7: Naye matches predict karna

Model ko **2D table** chahiye, chahe ek hi row ho (Lesson 1.3 ka `reshape(1, -1)` yaad karein). DataFrame se **column naam** bhi saath jate hain. FastAPI mein bilkul yahi line likhenge.

In [15]:
new_matches = pd.DataFrame({
    "runs_10":    [70, 100, 85],
    "wickets_10": [4,   1,   2],
})
new_matches["predicted_final"] = model.predict(new_matches).round()
new_matches

,runs_10,wickets_10,predicted_final
0,70,4,119.0
1,100,1,194.0
2,85,2,161.0


70/4 → kam total, 100/1 → bara total. Cricket ke hisaab se bilkul sense banta hai ✅

---
## Part 8: Residual analysis — model ki ghaltiyon se seekhna

**Residual (error) = actual − predicted**
- **plus** → model ne **kam** predict kiya
- **minus** → model ne **zyada** predict kiya

Model ko `venue` nazar nahi aa raha. Kya uski ghaltiyan venue ke hisaab se **ek taraf jhuki** hain?

In [16]:
comparison["venue"] = X_test["venue"]
comparison["error"] = comparison["actual"] - comparison["predicted"]
comparison.head()

,actual,predicted,venue,error
203,176,165.0,Rawalpindi,11.0
266,100,114.0,Lahore,-14.0
152,149,173.0,Multan,-24.0
9,96,95.0,Karachi,1.0
233,140,153.0,Multan,-13.0


In [17]:
comparison.groupby("venue")["error"].mean().round(1)

venue
Karachi        0.9
Lahore        -3.2
Multan       -11.4
Rawalpindi    14.3
Name: error, dtype: float64

### Natija

Model ne sab venues ka **average bonus (~+3)** intercept mein jor diya. To har venue ka bonus average se kitna door hai, wahi residual banta hai:

| Venue | Asal bonus − 3 | Residual |
|---|---|---|
| Rawalpindi | +15 − 3 = **+12** | ~**+14** (model **kam** predict karta hai) |
| Multan | −8 − 3 = **−11** | ~**−11** (model **zyada** predict karta hai) |
| Karachi | +5 − 3 = **+2** | ~**+1** |
| Lahore | 0 − 3 = **−3** | ~**−3** |

**Model ki ghaltiyon ne bata diya ke kaunsa feature gayab hai!** Jab ghaltiyan kisi group mein ek hi taraf jhuki hon, us group ko model ko sikhana hai.

---
## Part 9: Venue seedha daal dein? Error!

In [18]:
try:
    model_bad = LinearRegression()
    model_bad.fit(X_train, y_train)   # X_train mein 'venue' text hai
except ValueError as e:
    print("ValueError:", e)

ValueError: could not convert string to float: 'Karachi'


LinearRegression andar se `w × x` karta hai. `1.65 × "Rawalpindi"` ka koi matlab nahi. **Model sirf numbers samajhta hai.**

(Tip: lamba error ho to **sab se aakhri line** padhein, asal baat wahin hoti hai.)

---
## Part 10: OneHotEncoder — text ko numbers mein

**❌ Ghalat:** Karachi=1, Lahore=2, Multan=3, Rawalpindi=4 → model sochega Rawalpindi = 4 × Karachi. Venues ka koi **order** nahi (nominal).

**✅ Sahi: One-hot** — har venue ka **apna column**, sirf 0 ya 1:
```
venue        →  Karachi  Lahore  Multan  Rawalpindi
Rawalpindi   →     0       0       0         1
Karachi      →     1       0       0         0
```
Har row mein **sirf ek 1** ("one-hot").

| Parameter | Matlab |
|---|---|
| `sparse_output=False` | normal table ki shakal mein result |
| `handle_unknown="ignore"` | naya venue aaye (jo training mein nahi tha) to error nahi, saare 0 |
| `fit(X_train[["venue"]])` | **sirf training** se categories seekho. Double brackets (2D chahiye) |

In [19]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoder.fit(X_train[["venue"]])

print(encoder.categories_)
print(encoder.get_feature_names_out())

[array(['Karachi', 'Lahore', 'Multan', 'Rawalpindi'], dtype=object)]
['venue_Karachi' 'venue_Lahore' 'venue_Multan' 'venue_Rawalpindi']


### transform: text → 0/1
Dono outputs saath rakh ke dekhein: Rawalpindi wali row mein Rawalpindi wale (aakhri) column mein 1 hona chahiye.

In [20]:
train_venue = encoder.transform(X_train[["venue"]])

print(X_train["venue"].head())
print(train_venue[:5])

232       Karachi
59     Rawalpindi
6         Karachi
185        Lahore
173    Rawalpindi
Name: venue, dtype: str
[[1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]]


---
## Part 11: Numbers + encoded venue jor ke naya X

`prepare()` function taake train aur test **dono pe bilkul same** kaam ho.

| Hissa | Matlab |
|---|---|
| `encoder.transform(...)` | sirf **transform**, fit pehle ho chuka |
| `index=X.index` | naye columns ko **wahi index**, taake rows aligned rahein |
| `pd.concat([...], axis=1)` | do tables **side by side** jor do |

Result: 2 numeric + 4 venue = **6 columns**

In [21]:
def prepare(X):
    venue_encoded = encoder.transform(X[["venue"]])
    venue_df = pd.DataFrame(
        venue_encoded,
        columns=encoder.get_feature_names_out(),
        index=X.index,
    )
    return pd.concat([X[num_cols], venue_df], axis=1)

X_train_ready = prepare(X_train)
X_test_ready = prepare(X_test)

X_train_ready.head()

,runs_10,wickets_10,venue_Karachi,venue_Lahore,venue_Multan,venue_Rawalpindi
232,63,2,1.0,0.0,0.0,0.0
59,96,1,0.0,0.0,0.0,1.0
6,71,1,1.0,0.0,0.0,0.0
185,68,3,0.0,1.0,0.0,0.0
173,81,4,0.0,0.0,0.0,1.0


---
## Part 12: Naya model, venue ke saath (same test set pe comparison)

In [22]:
model2 = LinearRegression()
model2.fit(X_train_ready, y_train)

y_pred2 = model2.predict(X_test_ready)

print(f"MAE:     {mean_absolute_error(y_test, y_pred2):.2f}   (pehle {mae:.2f})")
print(f"RMSE:    {np.sqrt(mean_squared_error(y_test, y_pred2)):.2f}   (pehle {rmse:.2f})")
print(f"Test R²: {r2_score(y_test, y_pred2):.3f}  (pehle {r2:.3f})")

MAE:     9.99   (pehle 13.00)
RMSE:    12.93   (pehle 15.50)
Test R²: 0.845  (pehle 0.777)


### Natija
MAE ~13 → **~10** (~23% behtar), R² ~0.78 → **~0.85**.

**RMSE ~12.9, 0 kyun nahi?** Data mein `noise = rng.normal(0, 12, n)` tha: ±12 runs ki be-tarteebi jo **koi feature** predict nahi kar sakta. To RMSE ~12 se neeche ja hi nahi sakta. Model **lagbhag perfect** hai, bacha error sirf qudrati shor hai.

Real projects mein ye hadd **pata nahi hoti**, aur yahi sab se mushkil sawal hota hai: *"Model aur behtar ho sakta hai, ya ye data ki hadd hai?"*

---
## Part 13: Weights padhna

In [23]:
pd.Series(model2.coef_, index=X_train_ready.columns).round(2)

runs_10              1.69
wickets_10          -8.93
venue_Karachi        0.06
venue_Lahore        -5.07
venue_Multan        -8.75
venue_Rawalpindi    13.76
dtype: float64

| Feature | Asal | Model |
|---|---|---|
| runs_10 | 1.7 | ~1.69 ✅ |
| wickets_10 | −8 | ~−8.9 ✅ |

### Venue weights
| Venue | Asal bonus | Model |
|---|---|---|
| Rawalpindi | +15 | ~+13.8 |
| Karachi | +5 | ~+0.1 |
| Lahore | 0 | ~−5.1 |
| Multan | −8 | ~−8.8 |

Numbers exact nahi, lekin **order bilkul sahi** (Rawalpindi > Karachi > Lahore > Multan).

**Kyun?** One-hot mein har row mein sirf ek venue 1 hota hai, to model saare venue weights ko upar/neeche **khiska** ke farq intercept mein daal sakta hai. Isliye venue weights ko **aapas ke farq** se padhein:
```
Karachi − Lahore:  asal +5  →  model ~+5.1  ✅
```
Client ke liye: *"Rawalpindi ki pitch ~14 runs zyada deti hai"*. Linear model ki ye **samajh mein aane wali** khoobi clients ko bohat pasand aati hai.

---
## Part 14: Naya venue jo kabhi dekha nahi

In [24]:
encoder.transform(pd.DataFrame({"venue": ["Peshawar"]}))

array([[0., 0., 0., 0.]])

Saare **0** → "in 4 mein se koi nahi". `handle_unknown="ignore"` ki wajah se error nahi aaya.

**Real app ke liye zaroori:** FastAPI form mein koi "Peshawar" ya ghalti se "karachi" likhe to app **crash nahi** hogi; model "average venue" maan ke andaza dega.

---
## Khulasa

```
1. Data samjho → columns ka naqsha → leakage check
2. X (double [[ ]]) aur y (single [ ])
3. train_test_split → shapes print karo
4. model.fit(X_train, y_train) → model.predict(X_test)
5. MAE / RMSE / R² (train vs test compare)
6. Residuals ko groupby karo → gayab feature dhoondo
7. Text columns → OneHotEncoder (fit SIRF X_train pe)
8. Weights padho → model ne kya seekha
```

**Agla masla:** encoder fit, `prepare()`, transform train, transform test, concat, index... ye sab **haath se** karna padta hai, aur FastAPI mein dobara. Ek qadam bhoole to leakage ya ghalat columns. Iska hal: **Pipeline** (aage).